In [ ]:
import os
import sys
import subprocess
import json
import requests
import time

from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic", "tenacity"], check=True)

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"
BROKEN_URL = "http://localhost:9999"  # For failure testing

assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Live URL: {LIVE_URL}")
print(f"Broken URL (for tests): {BROKEN_URL}")
print("✅ Cell 1a complete. Environment is ready.")


Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Live URL: https://teachmewrite-ai-195673983724.asia-south1.run.app
Broken URL (for tests): http://localhost:9999
✅ Cell 1a complete. Environment is ready.


In [ ]:
from anthropic import Anthropic
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# --- Resilient API caller (retries with exponential backoff) ---
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type((requests.exceptions.RequestException, requests.exceptions.HTTPError)),
    reraise=False  # Return None on final failure instead of raising
)
def resilient_api_call(service_url: str, payload: dict) -> dict:
    """Call /feedback with retries. Raises on failure (caught by @retry). Returns None after 3 failed attempts."""
    print(f"    [Attempt] POST {service_url}/feedback")
    r = requests.post(f"{service_url}/feedback", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

# --- Tool implementations (wrap resilient caller with graceful failure) ---
def get_writing_feedback(student_text: str, level: int, service_url: str = None) -> dict:
    url = service_url or LIVE_URL
    print(f"  [Tool] get_writing_feedback (url={url})")
    try:
        result = resilient_api_call(url, {"student_text": student_text, "level": level})
        if result is None:
            return {"error": "Service unavailable after 3 retries. Please try again later."}
        return result
    except Exception as e:
        return {"error": f"Unexpected failure: {e}"}

def get_rewrite_suggestion(student_text: str, level: int, service_url: str = None) -> dict:
    url = service_url or LIVE_URL
    print(f"  [Tool] get_rewrite_suggestion (url={url})")
    try:
        result = resilient_api_call(url, {"student_text": student_text, "level": level})
        if result is None:
            return {"error": "Service unavailable after 3 retries. Please try again later."}
        return {"rewrite_suggestion": result.get("rewrite_suggestion", "")}
    except Exception as e:
        return {"error": f"Unexpected failure: {e}"}

print("✅ Cell 1b complete. Resilient tool wrappers ready.")

✅ Cell 1b complete. Resilient tool wrappers ready.


In [ ]:
# Minimal tools schema for this exercise (2 tools only, for focus)
tools = [
    {
        "name": "get_writing_feedback",
        "description": "Get structured writing feedback. Use when the student wants to understand their mistakes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    },
    {
        "name": "get_rewrite_suggestion",
        "description": "Get ONLY the corrected version of the text. Use when the student says 'just fix it'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    }
]

class ResilientAgent:
    def __init__(self, service_url: str = LIVE_URL):
        self.service_url = service_url
        self.system_prompt = """You are an English writing tutor for Pakistani students.

You have two tools:
1. `get_writing_feedback` — structured feedback for "help me understand"
2. `get_rewrite_suggestion` — corrected text for "just fix it"

CRITICAL — ERROR HANDLING:
If a tool returns a dict with an "error" key (not a normal response), the service is temporarily down.
In that case, you MUST:
- Not retry the same tool again.
- Politely tell the user that the service is temporarily unavailable.
- Suggest they try again in a minute.
Do NOT pretend the error didn't happen. Do NOT invent fake feedback.

Be brief, encouraging, and Socratic when explaining.
"""

    def chat(self, user_message: str, level: int = 5, max_iterations: int = 5) -> dict:
        messages = [{"role": "user", "content": user_message}]
        tools_called = []
        errors_seen = []

        for iteration in range(max_iterations):
            print(f"  [Iteration {iteration + 1}]")
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1024,
                system=self.system_prompt,
                tools=tools,
                messages=messages
            )

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        tool_name = block.name
                        tools_called.append(tool_name)
                        print(f"  [Agent] Tool: {tool_name}")

                        if tool_name == "get_writing_feedback":
                            result = get_writing_feedback(
                                service_url=self.service_url, **block.input
                            )
                        elif tool_name == "get_rewrite_suggestion":
                            result = get_rewrite_suggestion(
                                service_url=self.service_url, **block.input
                            )
                        else:
                            result = {"error": f"Unknown tool: {tool_name}"}

                        if "error" in result:
                            errors_seen.append(result["error"])

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })

                messages.append({"role": "user", "content": tool_results})

            elif response.stop_reason == "end_turn":
                final_text = "".join(b.text for b in response.content if hasattr(b, "text"))
                return {"response": final_text, "tools_called": tools_called, "errors": errors_seen}

            else:
                return {"response": f"Unexpected stop_reason: {response.stop_reason}", "tools_called": tools_called, "errors": errors_seen}

        return {"response": "Max iterations reached.", "tools_called": tools_called, "errors": errors_seen}

print("✅ Cell 1c complete. ResilientAgent is ready.")

✅ Cell 1c complete. ResilientAgent is ready.


In [ ]:
student_text = "I am agree with my father that education is more better than money."
query = f"{student_text} — Help me understand my mistakes."

# ============ TEST 1: BROKEN URL ============
print("=" * 60)
print("TEST 1: Broken URL (expect graceful failure)")
print("=" * 60)
broken_agent = ResilientAgent(service_url=BROKEN_URL)
r1 = broken_agent.chat(query, level=5)
print(f"\nTools called: {r1['tools_called']}")
print(f"Errors seen: {len(r1['errors'])}")
print(f"Response:\n{r1['response']}")
assert len(r1["errors"]) > 0, "Test 1 FAILED: agent should have seen errors"
print("\n✅ Test 1 passed: agent saw errors and responded gracefully.\n")


# ============ TEST 2: LIVE URL ============
print("=" * 60)
print("TEST 2: Live URL (expect normal operation)")
print("=" * 60)
live_agent = ResilientAgent(service_url=LIVE_URL)
r2 = live_agent.chat(query, level=5)
print(f"\nTools called: {r2['tools_called']}")
print(f"Errors seen: {len(r2['errors'])}")
print(f"Response:\n{r2['response']}")
assert len(r2["errors"]) == 0, f"Test 2 FAILED: agent should have no errors, got: {r2['errors']}"
print("\n✅ Test 2 passed: live service worked normally.\n")


# ============ TEST 3: INTERMITTENT FAILURE ============
print("=" * 60)
print("TEST 3: Intermittent failure (2 fail, then succeed)")
print("=" * 60)

original_post = requests.post
call_count = {"n": 0}

def flaky_post(*args, **kwargs):
    call_count["n"] += 1
    if call_count["n"] <= 2:
        print(f"    [Mock] Simulated network failure #{call_count['n']}")
        raise requests.exceptions.ConnectionError("Simulated network error")
    print(f"    [Mock] Success on attempt #{call_count['n']}")
    return original_post(*args, **kwargs)

# Patch requests.post for the retry test
requests.post = flaky_post
try:
    flaky_agent = ResilientAgent(service_url=LIVE_URL)
    r3 = flaky_agent.chat(query, level=5)
finally:
    requests.post = original_post

print(f"\nTools called: {r3['tools_called']}")
print(f"Errors seen: {len(r3['errors'])}")
print(f"Response:\n{r3['response']}")
assert call_count["n"] >= 3, f"Test 3 FAILED: retry logic didn't fire (only {call_count['n']} attempts)"
assert len(r3["errors"]) == 0, f"Test 3 FAILED: should recover with 0 errors, got: {r3['errors']}"
print("\n✅ Test 3 passed: retry logic recovered from transient failures.\n")

print("=" * 60)
print("✅ Cell 1d complete. All 3 resilience tests passed.")
print("=" * 60)

TEST 1: Broken URL (expect graceful failure)
  [Iteration 1]
  [Agent] Tool: get_writing_feedback
  [Tool] get_writing_feedback (url=http://localhost:9999)
    [Attempt] POST http://localhost:9999/feedback
    [Attempt] POST http://localhost:9999/feedback
    [Attempt] POST http://localhost:9999/feedback
  [Iteration 2]

Tools called: ['get_writing_feedback']
Errors seen: 1
Response:
I'm sorry! The feedback service is temporarily unavailable. Please try again in a minute.

In the meantime, here are **two quick things** to look for in your sentence:

1. **"I am agree"** — Think about what verb comes after "I am"? (Hint: it should be a verb ending in -ing, not just "agree")

2. **"more better"** — This is a **double comparison**. When a word already means "more" (like "better"), you don't need to add "more" before it.

Once the service is back up, I can give you detailed feedback. Good luck! 📚

✅ Test 1 passed: agent saw errors and responded gracefully.

TEST 2: Live URL (expect normal o

In [ ]:
artifact = {
    "test_date": "2026-10-22",
    "tests": [
        {
            "name": "broken_url",
            "result": "graceful_failure",
            "retries_observed": 3,
            "agent_response": r1["response"],
            "errors_seen": r1["errors"]
        },
        {
            "name": "live_url",
            "result": "normal_operation",
            "retries_observed": 0,
            "agent_response": r2["response"],
            "errors_seen": r2["errors"]
        },
        {
            "name": "intermittent_failure",
            "result": "recovered_via_retry",
            "attempts_until_success": call_count["n"],
            "agent_response": r3["response"],
            "errors_seen": r3["errors"]
        }
    ],
    "verification": "All 3 resilience tests passed",
    "retry_library": "tenacity",
    "retry_config": {"max_attempts": 3, "backoff": "exponential (1s, 2s, 4s)"}
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/resilience_day94.json"
with open(save_path, "w") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved resilience artifact to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 94 artifact saved.")


Saved resilience artifact to /content/drive/MyDrive/teachmewrite-ai/resilience_day94.json
✅ Day 94 artifact saved.
